# Serve Qwen3-8B (vLLM + tunnel)

OpenAI-compatible API for agent trajectory generation. See [docs/setup.md](https://github.com/abdelmagid07/latent_failiure_prediction/blob/main/docs/setup.md).

**Runtime:** A100 GPU. Keep this tab open while running `devbugs_agent_colab.ipynb`.

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


In [ ]:
# 1. Install a CUDA-12 build of vLLM.
import subprocess, sys

!pip -q uninstall -y vllm
!pip -q install "vllm==0.11.0"
# Colab preinstalls a newer transformers that removed all_special_tokens_extended,
# which vLLM 0.11.0's tokenizer-caching shim still calls. Force the compatible version.
!pip -q install --force-reinstall "transformers==4.55.4"
# vLLM's numba dependency (used for n-gram spec decoding) requires NumPy<=2.2.
!pip -q install "numpy<2.3"

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 \
  -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version

check = subprocess.run(
    [sys.executable, "-c", "import vllm._C; import vllm; print('vLLM', vllm.__version__, 'C-ext OK')"],
    capture_output=True, text=True,
)
print(check.stdout or check.stderr)
assert check.returncode == 0, "vLLM C extension failed to load — see error above"
print(f'Will invoke vLLM via: {sys.executable} -m vllm.entrypoints.openai.api_server')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.2/438.2 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.0/180.0 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 73.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 64.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━ 290.2/887.9 MB 99.4 MB/s eta 0:00:07

In [3]:
# CHANGE MODEL HERE -- FOR REAL EXPERIMENTS MUST BE AS BELOW:
# MODEL = "Qwen/Qwen3-8B"
# SERVED_NAME = "Qwen3-8B

# This is only for T4
MODEL = 'Qwen/Qwen3-1.7B'
SERVED_NAME = 'Qwen3-1.7B'

# This is same regardless of gpu, dont need api key for now
MAX_MODEL_LEN = 8192  # drop from 32768 on T4
PORT = 8000
API_KEY = "EMPTY"



In [4]:
# 3. Launch vLLM as a Python module (bypasses the stale system CLI binary).
#
# THINKING IS ON.
#
# The reasoning parser below splits the generated <think> block into a separate
# reasoning_content field so the hermes tool parser can still populate tool_calls
# (thinking-on + tool calls otherwise leaves the tool call unparsed inside the
# <think> text — see vllm-project/vllm#20611). stage2.extract.project_steps
# re-renders reasoning_content + tool_calls through the chat template to reproduce
# the exact generated tokens.
import subprocess, sys, os, time


# FOR T4 GPU IT NEEDS --dtype as float16, BUT FOR A100 IT NEEDS bfloat16
log = open("vllm.log", "w")
cmd = [
    sys.executable, "-m", "vllm.entrypoints.openai.api_server",
    "--model", MODEL,
    "--served-model-name", SERVED_NAME,
    "--dtype", "float16",
    "--max-model-len", str(MAX_MODEL_LEN),
    "--port", str(PORT),

    # Required for mini-SWE-agent tool calls
    "--enable-auto-tool-choice",
    "--tool-call-parser", "hermes",
    # Splits the <think> block into reasoning_content so tool calls still parse
    # under thinking-on. Use "deepseek_r1" if this vLLM build lacks "qwen3".
    "--reasoning-parser", "qwen3",
]
if API_KEY and API_KEY != "EMPTY":
    cmd += ["--api-key", API_KEY]
print(" ".join(cmd))
vllm_proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)
print("vLLM starting (pid %d); first run downloads ~16GB of weights..." % vllm_proc.pid)


/usr/bin/python3 -m vllm.entrypoints.openai.api_server --model Qwen/Qwen3-0.6B --served-model-name Qwen3-0.6B --dtype float16 --max-model-len 8192 --port 8000 --enable-auto-tool-choice --tool-call-parser hermes --reasoning-parser qwen3
vLLM starting (pid 1709); first run downloads ~16GB of weights...


In [5]:
# 4. Wait until the server answers /v1/models (model load can take a few minutes).
import requests, time

headers = {} if API_KEY in ("", "EMPTY") else {"Authorization": f"Bearer {API_KEY}"}
url = f"http://localhost:{PORT}/v1/models"
for _ in range(120):  # up to ~10 min
    if vllm_proc.poll() is not None:
        raise RuntimeError("vLLM exited early — see tail below:\n" + open("vllm.log").read()[-3000:])
    try:
        r = requests.get(url, headers=headers, timeout=5)
        if r.ok:
            print("vLLM is up:", r.json())
            break
    except Exception:
        pass
    time.sleep(5)
else:
    raise TimeoutError("vLLM did not become ready; check vllm.log")


vLLM is up: {'object': 'list', 'data': [{'id': 'Qwen3-0.6B', 'object': 'model', 'created': 1784320170, 'owned_by': 'vllm', 'root': 'Qwen/Qwen3-0.6B', 'parent': None, 'max_model_len': 8192, 'permission': [{'id': 'modelperm-ff14f2c2b836430b96a33c93c14bbb5e', 'object': 'model_permission', 'created': 1784320170, 'allow_create_engine': False, 'allow_sampling': True, 'allow_logprobs': True, 'allow_search_indices': False, 'allow_view': True, 'allow_fine_tuning': False, 'organization': '*', 'group': None, 'is_blocking': False}]}]}


In [7]:
# only run this cell if prev cell failed, this is full logs for debugging
print(open('vllm.log').read()[-12000:])

 INFO 07-17 20:28:04 [core.py:644] Waiting for init message from front-end.
(EngineCore_DP0 pid=1935) INFO 07-17 20:28:04 [core.py:77] Initializing a V1 LLM engine (v0.11.0) with config: model='Qwen/Qwen3-0.6B', speculative_config=None, tokenizer='Qwen/Qwen3-0.6B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=8192, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='qwen3'), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_endpoint=None, collect_detailed_traces=None), seed=0, served_model_name=Qwen3-0.6B, 

In [8]:
import subprocess, re, time

tunnel_log = open('cloudflared.log', 'w')
tunnel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', f'http://localhost:{PORT}', '--no-autoupdate'],
    stdout=tunnel_log, stderr=subprocess.STDOUT,
)
public = None
for _ in range(60):
    time.sleep(2)
    if tunnel.poll() is not None:
        raise RuntimeError('cloudflared exited early:\n' + open('cloudflared.log').read()[-2000:])
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('cloudflared.log').read())
    if m:
        public = m.group(0)
        break
if not public:
    tunnel.terminate()
    raise TimeoutError('No trycloudflare URL after 120s:\n' + open('cloudflared.log').read()[-2000:])
print(f'MODEL_API_BASE={public}/v1')

MODEL_API_BASE=https://episode-marks-wheat-truly.trycloudflare.com/v1


In [9]:
import time
try:
    while True:
        time.sleep(30)
except KeyboardInterrupt:
    tunnel.terminate()
    vllm_proc.terminate()